# AthleAgent — Model Improvement & Selection Journey

**Walkthrough notebook** — ML analysis in one place: data, features, model comparison, winner selection, and operating threshold.

> **Before running:** `python ML_model/run_pipeline.py` (creates `artifacts/promoted.json` and comparison artifacts).  
> **Production contracts (no duplicate analysis):** `backend/docs/FEATURES.md`, `MODEL.md`.

---

## Notebook map (read top to bottom)

| Part | Topic | Key takeaway |
|------|-------|--------------|
| **0** | Prediction target | Risk for **day D (today)**, not tomorrow |
| **1** | Data creation | Hazard model + per-athlete simulation |
| **2** | EDA | Do ACWR / sleep / HRV align with the label? |
| **3** | Feature improvement | v0→final: 38→35, drop std21 + redundant chronic, Top-7 importance |
| **4** | Feature pipeline | ACWR, sleep debt, hrv_drop — same logic in train & serve |
| **5** | **Model selection** | **11 models + Recall-first policy + operating threshold** |
| **6** | Threshold & calibration | Why 0.18? sweep, calibration, UI bands |
| **7** | Evolution | v1 Raw @ 0.30 → final XGBoostDeep @ 0.18 |
| **8** | Summary | Metrics + deployment gates |

**Run:** Run All top to bottom. **Charts worth showing when explaining:** Chart A (Part 3), Recall–FPR (Part 5), Threshold sweep (Part 6).

## Part 0 — What we predict

| Layer | Meaning |
|-------|---------|
| **Product** | Morning of day **D** → **injury risk for today (D)**, not tomorrow |
| **Inputs** | Wearable + check-in + nutrition for **D**, plus up to **7 days** of history (through **D−1**) |
| **Training column** | `injury_today` — row date = **D**; label = **injury on day D** (0/1, training only) |
| **Firestore** | `injuredYesterday` on check-in **D+1** reports injury on **day D** (dataset building only) |

> **Why this matters for model selection:** Our policy prioritizes high Recall — in prevention, a false alarm is preferable to missing a risky day.

In [1]:
# pip install pandas numpy matplotlib jupyter  (one-time)
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

NOTEBOOK_DIR = Path.cwd().resolve()
ML_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR / "ML_model"
PROJECT_ROOT = ML_ROOT.parent
ARTIFACTS_ROOT = ML_ROOT / "artifacts"
LABEL_COL = "injury_today"  # training label: injury on row date D

sys.path.insert(0, str(ML_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "backend"))

from data_generator import generate_synthetic_data, compute_training_reference_features  # noqa: E402
from services.model_features import MODEL_FEATURE_COLUMNS  # noqa: E402

promoted_path = ARTIFACTS_ROOT / "promoted.json"
if not promoted_path.exists():
    raise FileNotFoundError(f"Missing {promoted_path}. Run: python ML_model/run_pipeline.py")

promoted = json.loads(promoted_path.read_text(encoding="utf-8"))
if promoted.get("artifacts_dir"):
    RUN_DIR = Path(promoted["artifacts_dir"])
    if not RUN_DIR.is_absolute():
        RUN_DIR = PROJECT_ROOT / RUN_DIR
elif promoted.get("model_path"):
    model_path = Path(promoted["model_path"])
    RUN_DIR = (PROJECT_ROOT / model_path).parent if not model_path.is_absolute() else model_path.parent
else:
    raise KeyError("promoted.json must contain artifacts_dir or model_path")

WINNER = "XGBoostDeep"
manifest = json.loads((RUN_DIR / "run_manifest.json").read_text(encoding="utf-8"))
OPERATING_THRESHOLD = float(manifest["threshold"])
winner_metrics = manifest["winner_metrics"]
policy = manifest["policy"]
WINNER = str(manifest.get("winner") or WINNER)

DATA_CANDIDATES = [
    ML_ROOT / "athlete_injury_data.csv",
    PROJECT_ROOT / "research_archive" / "ML_model_root_outputs" / "athlete_injury_data.csv",
]

print(f"Project: {PROJECT_ROOT.name}")
print(f"Promoted run: {manifest['run_id']} | winner: {manifest['winner']} | threshold: {OPERATING_THRESHOLD}")

Project: final_project_AthleAgent
Promoted run: 20260629_113445 | winner: XGBoostDeep | threshold: 0.18


---
## Snapshot — key numbers (after Setup)

Quick reference for explaining what was chosen and why.

In [ ]:

fi = pd.read_csv(RUN_DIR / "feature_importance.csv")
fi["importance_pct"] = (fi["importance"] / fi["importance"].sum() * 100).round(1)
fi["cumulative_pct"] = fi["importance_pct"].cumsum().round(1)
REMOVED_V0_STD21 = ["acwr_ratio_std21", "sleep_hours_std21"]
REMOVED_REDUNDANT = ["chronic_load_7d"]
REMOVED_FEATURES = REMOVED_V0_STD21 + REMOVED_REDUNDANT

display(Markdown("### Snapshot — AthleAgent ML"))
display(pd.DataFrame([
    ("Predicting", "P(injury on day D) — daily risk in the app"),
    ("Features in model", str(len(MODEL_FEATURE_COLUMNS))),
    ("Removed std21 (v0)", f"{len(REMOVED_V0_STD21)} — needed 21d app history"),
    ("Removed redundant load", f"{len(REMOVED_REDUNDANT)} — chronic_load_7d (ACWR baseline is internal)"),
    ("Selected model", manifest["winner"]),
    ("High-risk if score >=", f"score ≥ {OPERATING_THRESHOLD}"),
    ("Recall @ threshold", f"{winner_metrics['Recall@Threshold']:.1%}"),
    ("ROC-AUC", f"{winner_metrics['ROC-AUC']:.3f}"),
    ("Brier (calibration)", f"{winner_metrics['BrierScore']:.3f}"),
    ("#1 feature", f"{fi.iloc[0]['feature']} ({fi.iloc[0]['importance_pct']:.1f}%)"),
    ("Top-7 share of importance", f"{fi.head(7)['importance_pct'].sum():.1f}%"),
], columns=["Item", "Value"]))

display(Markdown("### Selection policy (from `run_manifest.json`)"))
display(pd.DataFrame([
    ("Hard Recall gate", f"≥ {policy['recall_hard_min']:.0%}"),
    ("Recall target", f"≥ {policy['recall_min']:.0%}"),
    ("Max FPR", f"≤ {policy['fpr_max_operating']:.0%}"),
    ("Min Precision", f"≥ {policy['precision_min']:.0%}"),
    ("Min F1", f"≥ {policy['f1_min']:.0%}"),
], columns=["Rule", "Value"]))

display(Markdown("### Top 7 features (learned importance — not hand-tuned weights)"))
display(
    fi.head(7)[["feature", "importance_pct", "cumulative_pct"]]
    .rename(columns={"feature": "Feature", "importance_pct": "% of total", "cumulative_pct": "% cumulative"})
    .style.format({"% of total": "{:.1f}%", "% cumulative": "{:.1f}%"})
    .background_gradient(subset=["% of total"], cmap="YlOrRd")
)



---
## Part 1 — Data creation (`data_generator.py`)

We do not yet have enough real injury labels for the full ~345k-row training set, so we train on **synthetic data** grounded in sports-science signals:

1. **Per-athlete simulation** — load, sleep, HRV, stress, nutrition evolve day by day.
2. **Logistic hazard model** — injury probability from ACWR excess, sleep debt, HRV drop, prior injury, etc.
3. **Stochastic label** — `injury_tomorrow` = 1 if random draw < hazard (injury **on that day**).
4. **Post-processing** — rolling ACWR, `sleep_debt_3d`, `hrv_drop`, `load_recovery_imbalance`.

**Note:** The hazard simulator is custom domain logic; the ML model learns to approximate it from the engineered features.

In [ ]:
MINI_ATHLETES, MINI_DAYS, SEED = 5, 120, 42
print(f"Generating {MINI_ATHLETES} athletes × {MINI_DAYS} days (seed={SEED})...")
df_mini = generate_synthetic_data(num_athletes=MINI_ATHLETES, days_per_athlete=MINI_DAYS, seed=SEED)
df_mini["date"] = pd.to_datetime(df_mini["date"])

summary = pd.Series({
    "rows": len(df_mini),
    "athletes": df_mini["athlete_id"].nunique(),
    "columns": df_mini.shape[1],
    "injury-day rate": df_mini[LABEL_COL].mean(),
    "date range": f"{df_mini['date'].min().date()} → {df_mini['date'].max().date()}",
})
display(summary.to_frame("value"))
display(df_mini.head(8))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df_mini[LABEL_COL].value_counts().sort_index().plot(
    kind="bar", ax=axes[0], color=["#95a5a6", "#e74c3c"], rot=0
)
axes[0].set_title("Label balance (mini dataset)")
axes[0].set_xticklabels(["No injury on D", "Injury on D"])

per_athlete = df_mini.groupby("athlete_id")[LABEL_COL].mean()
axes[1].bar(per_athlete.index.astype(str), per_athlete.values, color="#9b59b6")
axes[1].set_title("Injury-day rate per athlete")
axes[1].set_ylabel("Rate")
plt.tight_layout()
plt.show()

print("Production-scale target:", f"{manifest['dataset_rows']:,} rows (1000 athletes × 365 days)")

In [ ]:
# One athlete timeline: load + risk drivers + injury days
aid = int(df_mini["athlete_id"].iloc[0])
one = df_mini[df_mini["athlete_id"] == aid].sort_values("date")

fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
axes[0].plot(one["date"], one["daily_distance_km"], color="#2980b9", label="distance (km)")
axes[0].set_ylabel("km")
axes[0].set_title(f"Athlete {aid} — daily load")
axes[0].legend(loc="upper right")

axes[1].plot(one["date"], one["acwr_ratio"], label="ACWR", color="#e67e22")
axes[1].axhline(1.4, color="red", ls="--", alpha=0.6, label="risk band ~1.4")
axes[1].plot(one["date"], one["sleep_debt_3d"], label="sleep debt 3d", color="#8e44ad", alpha=0.8)
axes[1].set_ylabel("ratio / hours")
axes[1].legend(loc="upper right")
axes[1].set_title("Workload & recovery signals")

injury_days = one[one[LABEL_COL] == 1]["date"]
axes[2].plot(one["date"], one["hrv_drop"], color="#16a085", label="hrv_drop")
for d in injury_days:
    axes[2].axvline(d, color="#e74c3c", alpha=0.35, lw=2)
axes[2].set_ylabel("HRV drop")
axes[2].set_title("HRV drop (red bands = injury on day D)")
axes[2].legend(loc="upper right")
plt.tight_layout()
plt.show()

### Hazard drivers (why labels are not random)

The generator builds a **logistic hazard** from interpretable factors (ACWR excess, sleep debt, HRV stress, `injured_yesterday`, etc.), then samples the binary label.  
High-risk rows should correlate with those factors — we check that in EDA.

---
## Part 2 — EDA (sanity-check before modeling)

We inspect class balance, feature distributions, and **label vs driver** relationships.  
Uses the **full CSV** when available; otherwise the mini sample from Part 1.

> **Takeaway:** Higher ACWR → higher injury-day rate. That supports our feature set and Recall-first policy.

In [ ]:
full_path = next((p for p in DATA_CANDIDATES if p.exists()), None)
if full_path:
    print(f"Loading full dataset: {full_path}")
    df_eda = pd.read_csv(full_path, parse_dates=["date"])
    if len(df_eda) > 80_000:
        df_eda = df_eda.sample(80_000, random_state=SEED)  # faster plots
        print(f"Sampled {len(df_eda):,} rows for EDA plots")
else:
    print("Full CSV not found — using mini dataset for EDA")
    df_eda = df_mini.copy()

eda_cols = [
    "daily_distance_km", "sleep_hours", "stress_level", "acwr_ratio",
    "sleep_debt_3d", "hrv_drop", "load_recovery_imbalance", LABEL_COL,
]
display(df_eda[eda_cols].describe().T.round(3))

In [ ]:
# Correlation with injury-on-day-D
corr = df_eda[eda_cols].corr()[LABEL_COL].drop(LABEL_COL).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#e74c3c" if v > 0 else "#3498db" for v in corr.values]
ax.barh(corr.index, corr.values, color=colors)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel(f"Correlation with `{LABEL_COL}` (injury on day D)")
ax.set_title("Which signals align with the label?")
plt.tight_layout()
plt.show()
display(corr.to_frame("correlation").round(3))

In [ ]:
# Injury-day rate vs ACWR bins (classic workload monitor)
bins = pd.cut(df_eda["acwr_ratio"], bins=[0.35, 0.9, 1.15, 1.4, 1.8, 2.8])
by_acwr = df_eda.groupby(bins, observed=True)[LABEL_COL].agg(["mean", "count"])
by_acwr.columns = ["injury_day_rate", "n"]
display(by_acwr.round(3))

fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(by_acwr))
ax.bar(x, by_acwr["injury_day_rate"] * 100, color="#e67e22")
ax.set_xticks(x)
ax.set_xticklabels([str(i) for i in by_acwr.index], rotation=30, ha="right")
ax.set_ylabel("Injury-day rate (%)")
ax.set_title("Injury on day D vs ACWR bin")
plt.tight_layout()
plt.show()

In [ ]:
# Distribution snapshots
plot_feats = ["daily_distance_km", "sleep_hours", "stress_level", "hrv_drop", "load_recovery_imbalance"]
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
axes = axes.ravel()
for ax, col in zip(axes, plot_feats):
    df_eda[col].hist(ax=ax, bins=30, color="#3498db", edgecolor="white")
    ax.set_title(col)
axes[-1].axis("off")
plt.suptitle("Feature distributions (EDA sample)", y=1.02)
plt.tight_layout()
plt.show()

---
## Part 3 — Feature improvement (38 → 35)

| Output | What it explains |
|--------|------------------|
| **Table 1** | Early v0 (~38) vs final (35) — kept vs removed |
| **Table 2** | All 35 features: category, Firestore source, formula, importance % |
| **Charts A–B** | Importance ranking; sum by category |
| **Table 3 + Chart C** | Top-7: do high values mean more injuries? |
| **Chart D** | Correlation heatmap |
| **Table 4** | Raw vs engineered |
| **Section 3.2** | Version timeline (v0 → final) |

**Prerequisite:** run **Part 2** first so `df_eda` exists.

### Why we removed features
- `acwr_ratio_std21`, `sleep_hours_std21` — require **21 days of app history**; production only guarantees **7**.
- `chronic_load_7d` (legacy name `chronic_load_21d`) — **redundant** with `acute_load_7d` + `acwr_ratio` on a 7-day window; ACWR baseline is computed internally, not stored as a model column.

In [ ]:

REMOVED_V0_STD21 = ["acwr_ratio_std21", "sleep_hours_std21"]
REMOVED_REDUNDANT = ["chronic_load_7d"]
REMOVED_FEATURES = REMOVED_V0_STD21 + REMOVED_REDUNDANT
if "importance_pct" not in fi.columns:
    fi["importance_pct"] = (fi["importance"] / fi["importance"].sum() * 100).round(1)

FEATURE_CATALOG = [
    {"feature": "hrv_drop", "category": "History (7d)", "kind": "Engineered", "firestore": "hrvRmssd + 7d history", "formula": "HRV today - mean(HRV, 7d)"},
    {"feature": "stress_level", "category": "Subjective", "kind": "Direct", "firestore": "checkins.stressLevel", "formula": "Scale to 1-10"},
    {"feature": "load_recovery_imbalance", "category": "Derived", "kind": "Interaction", "firestore": "computed", "formula": "acwr_ratio * sleep_debt_3d"},
    {"feature": "injured_yesterday", "category": "State", "kind": "Direct", "firestore": "checkins.injuredYesterday", "formula": "Injury on day D-1"},
    {"feature": "acwr_ratio", "category": "History (7d)", "kind": "Engineered", "firestore": "distance + history", "formula": "acute_7d / internal baseline; clip 0.35-2.8"},
    {"feature": "history_injury_count", "category": "Profile", "kind": "Direct", "firestore": "users.historyInjuryCount", "formula": "From registration"},
    {"feature": "sleep_debt_3d", "category": "History (3d)", "kind": "Engineered", "firestore": "sleepMinutes + history", "formula": "sum(max(0, 8 - sleep)) over 3d"},
    {"feature": "daily_distance_km", "category": "Load", "kind": "Direct", "firestore": "health.distanceMeters", "formula": "m / 1000"},
    {"feature": "sleep_hours", "category": "Recovery", "kind": "Direct", "firestore": "health.sleepMinutes", "formula": "min / 60"},
    {"feature": "active_calories_burned", "category": "Load", "kind": "Direct", "firestore": "health.activeCalories", "formula": "As recorded"},
    {"feature": "workout_intensity_minutes", "category": "Load", "kind": "Derived", "firestore": "computed", "formula": "dist*5.5 + cal/40"},
    {"feature": "muscle_soreness", "category": "Subjective", "kind": "Direct", "firestore": "checkins.muscleSoreness", "formula": "Scale 1-5 to 1-10"},
    {"feature": "acute_load_7d", "category": "History", "kind": "Engineered", "firestore": "distance history", "formula": "7d mean distance km"},
    {"feature": "acwr_ratio_ma7", "category": "History", "kind": "Rolling", "firestore": "train/serve", "formula": "7d mean of ACWR"},
    {"feature": "energy_level", "category": "Subjective", "kind": "Direct", "firestore": "checkins.energyLevel", "formula": "Scale to 1-10"},
    {"feature": "bmi", "category": "Profile", "kind": "Derived", "firestore": "weightKg, heightCm", "formula": "weight / height^2"},
    {"feature": "age", "category": "Profile", "kind": "Direct", "firestore": "users.age", "formula": "Profile"},
    {"feature": "body_fat_pct", "category": "Profile", "kind": "Direct", "firestore": "health.bodyFatPct", "formula": "Wearable"},
    {"feature": "vo2_max", "category": "Profile", "kind": "Direct", "firestore": "health.vo2Max", "formula": "Wearable"},
    {"feature": "avg_cadence", "category": "Load", "kind": "Direct/fallback", "firestore": "health.avgCadence", "formula": "Sensor or derived"},
    {"feature": "elevation_gained_m", "category": "Load", "kind": "Direct", "firestore": "health.elevationGainedMeters", "formula": "Sensor"},
    {"feature": "floors_climbed", "category": "Load", "kind": "Direct", "firestore": "health.floorsClimbed", "formula": "Sensor"},
    {"feature": "avg_speed", "category": "Load", "kind": "Direct/fallback", "firestore": "health.avgSpeed", "formula": "Sensor or derived"},
    {"feature": "max_speed", "category": "Load", "kind": "Direct/fallback", "firestore": "health.maxSpeed", "formula": "Sensor or derived"},
    {"feature": "avg_power", "category": "Load", "kind": "Direct", "firestore": "health.avgPower", "formula": "Power meter"},
    {"feature": "hrv_score", "category": "Recovery", "kind": "Direct", "firestore": "health.hrvRmssd", "formula": "RMSSD"},
    {"feature": "resting_hr", "category": "Recovery", "kind": "Direct", "firestore": "restingHeartRate", "formula": "Priority chain in API"},
    {"feature": "respiratory_rate", "category": "Recovery", "kind": "Direct", "firestore": "health.respiratoryRate", "formula": "Sensor"},
    {"feature": "spo2", "category": "Recovery", "kind": "Direct", "firestore": "health.oxygenSaturation", "formula": "Sensor"},
    {"feature": "nutrition_intake_calories", "category": "Nutrition", "kind": "Direct", "firestore": "nutrition.totalCalories", "formula": "Food intake"},
    {"feature": "daily_calories", "category": "Nutrition", "kind": "Derived", "firestore": "nutrition macros", "formula": "Derived intake"},
    {"feature": "total_calories_burned", "category": "Energy", "kind": "Derived", "firestore": "active + BMR", "formula": "Burn not intake"},
    {"feature": "calorie_balance", "category": "Energy", "kind": "Derived", "firestore": "computed", "formula": "intake - burn"},
    {"feature": "sleep_hours_ma7", "category": "History", "kind": "Rolling", "firestore": "train/serve", "formula": "7d mean sleep"},
    {"feature": "speed_intensity_ratio", "category": "Derived", "kind": "Interaction", "firestore": "computed", "formula": "max_speed / (avg_speed+0.1)"},
]
catalog_df = pd.DataFrame(FEATURE_CATALOG).merge(fi[["feature", "importance_pct"]], on="feature")
catalog_df["rank"] = catalog_df["importance_pct"].rank(ascending=False, method="min").astype(int)
catalog_df = catalog_df.sort_values("rank")

final_set = set(MODEL_FEATURE_COLUMNS)
v0_set = final_set | set(REMOVED_FEATURES)

def _removal_status(feat: str) -> str:
    if feat in REMOVED_V0_STD21:
        return "REMOVED — needs 21d history"
    if feat in REMOVED_REDUNDANT:
        return "REMOVED — redundant (acute + ACWR only)"
    return "Kept"

t1_rows = []
for feat in sorted(v0_set):
    t1_rows.append({
        "Feature": feat,
        "In early v0 (~38)": "Yes",
        f"In final model ({len(final_set)})": "Yes" if feat in final_set else "No",
        "Status": _removal_status(feat),
    })
t1 = pd.DataFrame(t1_rows).sort_values(["Status", "Feature"])

display(Markdown("### Table 1 — Feature set evolution (v0 → final)"))
def _row_color(row):
    if "REMOVED" in row["Status"]:
        return ["background-color:#ffdddd"] * len(row)
    return ["background-color:#ddffdd"] * len(row)
display(t1.style.apply(_row_color, axis=1))

display(Markdown(
    f"**Takeaway:** Production uses **{len(final_set)}** features. "
    f"Removed **{len(REMOVED_V0_STD21)}** std21 stats (21d history) and **{len(REMOVED_REDUNDANT)}** redundant chronic load column."
))

display(Markdown(f"### Table 2 — Complete catalog ({len(final_set)} features, sorted by importance)"))
t2 = catalog_df.rename(columns={
    "rank": "Rank", "feature": "Feature", "category": "Category", "kind": "Type",
    "importance_pct": "Importance %", "firestore": "Firestore field", "formula": "Formula / logic",
})[["Rank", "Feature", "Category", "Type", "Importance %", "Firestore field", "Formula / logic"]]
display(t2.style.format({"Importance %": "{:.1f}%"}).background_gradient(subset=["Importance %"], cmap="YlOrRd"))

display(Markdown(f"### Chart A — Importance of all {len(final_set)} features"))
fig, ax = plt.subplots(figsize=(10, 11))
plot_fi = fi.sort_values("importance_pct", ascending=True)
bar_colors = ["#c0392b" if f in fi.head(7)["feature"].values else "#2980b9" for f in plot_fi["feature"]]
ax.barh(plot_fi["feature"], plot_fi["importance_pct"], color=bar_colors)
top7_sum = fi.head(7)["importance_pct"].sum()
ax.axvline(top7_sum, color="green", ls="--", lw=2, label=f"Top-7 combined = {top7_sum:.1f}%")
ax.set_xlabel("Share of total XGBoost importance (%)")
ax.set_title("Red bars = top 7 drivers (~55% of model)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

display(Markdown("### Chart B — Importance by category (which data source matters?)"))
by_cat = catalog_df.groupby("category")["importance_pct"].sum().sort_values()
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(by_cat.index, by_cat.values, color="#8e44ad")
for i, v in enumerate(by_cat.values):
    ax.text(v + 0.2, i, f"{v:.1f}%", va="center")
ax.set_xlabel("Total importance (%)")
plt.tight_layout()
plt.show()

display(Markdown("### Table 3 + Chart C — Do higher values mean more injuries? (top 7 features)"))
TOP7 = fi.head(7)["feature"].tolist()
rate_rows = []
lo_vals, hi_vals, labels = [], [], []
for feat in TOP7:
    if feat not in df_eda.columns:
        continue
    q33, q67 = df_eda[feat].quantile([0.33, 0.67])
    r_lo = df_eda.loc[df_eda[feat] <= q33, LABEL_COL].mean()
    r_hi = df_eda.loc[df_eda[feat] >= q67, LABEL_COL].mean()
    rate_rows.append({"Feature": feat, "Lowest third": f"{r_lo:.1%}", "Highest third": f"{r_hi:.1%}", "Direction": "Higher -> more injuries" if r_hi > r_lo else "Check relationship"})
    lo_vals.append(r_lo * 100)
    hi_vals.append(r_hi * 100)
    labels.append(feat)
display(pd.DataFrame(rate_rows))

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(labels))
ax.bar(x - 0.2, lo_vals, 0.4, label="Lowest third of athletes", color="#27ae60")
ax.bar(x + 0.2, hi_vals, 0.4, label="Highest third of athletes", color="#c0392b")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha="right")
ax.set_ylabel("Injury-day rate (%)")
ax.set_title("Injury rate when feature is low vs high")
ax.legend()
plt.tight_layout()
plt.show()

display(Markdown("### Chart D — Correlation heatmap (top 12 features + injury label)"))
heat_cols = [c for c in fi.head(12)["feature"].tolist() + [LABEL_COL] if c in df_eda.columns]
corr_mat = df_eda[heat_cols].corr()
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_mat.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(heat_cols)))
ax.set_yticks(range(len(heat_cols)))
ax.set_xticklabels(heat_cols, rotation=45, ha="right")
ax.set_yticklabels(heat_cols)
plt.colorbar(im, ax=ax, label="Pearson correlation")
ax.set_title("Last row/column = injury on day D (label)")
plt.tight_layout()
plt.show()

display(Markdown("### Table 4 — Raw inputs vs engineered features"))
kind_tbl = catalog_df["kind"].value_counts().reset_index()
kind_tbl.columns = ["Processing type", "Number of features"]
display(kind_tbl)
fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(kind_tbl["Number of features"], labels=kind_tbl["Processing type"], autopct="%1.0f%%", startangle=140)
ax.set_title("Most features are engineered or rolling, not raw sensors")
plt.tight_layout()
plt.show()



---
## Part 4 — How features are built (pipeline)

After Part 3 (what goes into the model), this part shows **how** rolling features are computed on one athlete timeline.  
**Same logic** in `data_generator.py` (training) and `backend/services/history/rolling_features.py` (production).

### 4.1 Pipeline — data path

```
Daily signals (distance, sleep, HRV, check-in, nutrition, profile)
    → rolling windows (7d acute load, internal ACWR baseline, 3d sleep debt, 7d HRV baseline)
    → ratios & interactions (ACWR, load_recovery_imbalance, speed_intensity_ratio)
    → sequential MA7 features (train_model.add_sequential_features)
    → 35 columns in MODEL_FEATURE_COLUMNS (served in backend)
```

### 4.2 Core formulas

| Feature | Formula (intuition) |
|---------|---------------------|
| `acute_load_7d` | 7-day mean distance (km) |
| `acwr_ratio` | acute / **internal baseline** (`mean×0.85 + std×0.35 + 0.5`); baseline is **not** a model feature; clipped 0.35–2.8 |
| `sleep_debt_3d` | Σ max(0, 8 − sleep_hours) over last 3 days |
| `hrv_drop` | today HRV − 7-day mean HRV, clipped ±15 |
| `load_recovery_imbalance` | `acwr_ratio × sleep_debt_3d` |
| `calorie_balance` | intake − burned |
| `speed_intensity_ratio` | max_speed / (avg_speed + 0.1) |

In [ ]:
# Step-by-step on synthetic histories (train/serve reference helper)
hist = df_mini[df_mini["athlete_id"] == aid].sort_values("date")
dist_hist = hist["daily_distance_km"].tolist()
sleep_hist = hist["sleep_hours"].tolist()
hrv_hist = hist["hrv_score"].tolist()

steps = []
for day_idx in range(6, min(25, len(hist))):
    ref = compute_training_reference_features(
        dist_hist[: day_idx + 1],
        sleep_hist[: day_idx + 1],
        hrv_hist[: day_idx + 1],
    )
    row = hist.iloc[day_idx]
    steps.append({
        "date": row["date"].date(),
        "acute_load_7d": ref["acute_load_7d"],
        "acwr_ratio": ref["acwr_ratio"],
        "sleep_debt_3d": ref["sleep_debt_3d"],
        "hrv_drop": ref["hrv_drop"],
        "load_recovery_imbalance": row["load_recovery_imbalance"],
        LABEL_COL: int(row[LABEL_COL]),
    })
steps_df = pd.DataFrame(steps)
display(steps_df.round(3))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps_df["date"], steps_df["acwr_ratio"], "o-", label="ACWR")
ax2 = ax.twinx()
ax2.bar(steps_df["date"], steps_df["load_recovery_imbalance"], alpha=0.35, color="#8e44ad", label="load×sleep debt")
ax.set_title("Feature build-up over early season (one athlete)")
ax.set_ylabel("ACWR")
ax2.set_ylabel("load_recovery_imbalance")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Feature groups → data source (production contract)
feature_groups = pd.DataFrame([
    ("Profile", "age, bmi, body_fat_pct, vo2_max, history_injury_count", "users/{uid}"),
    ("Wearable day D", "sleep, distance, HR/HRV, calories, speed, power…", "daily_health/{D}"),
    ("Subjective day D", "stress, soreness, energy, injured_yesterday", "daily_checkins/{D}"),
    ("Nutrition day D", "intake calories, macros", "daily_nutrition/{D}"),
    ("History ≤ D−1", "ACWR, sleep debt, hrv_drop, MA7…", "7-day rolling"),
    ("Derived", "load_recovery_imbalance, calorie_balance, speed_intensity_ratio", "computed"),
], columns=["Group", "Examples", "Source"])
display(feature_groups)

print(f"\nFinal model uses {len(MODEL_FEATURE_COLUMNS)} features:")
for i, c in enumerate(MODEL_FEATURE_COLUMNS, 1):
    print(f"  {i:2d}. {c}")

### 3.2 — Improvement timeline (features + model)

| Stage | What changed | Why |
|-------|--------------|-----|
| **v0** | ~38 features incl. std21 | Removed — needs 21 days of app history |
| **v1** | XGBoostRaw, threshold 0.30 | High recall but FPR ~79%, weak calibration |
| **v2** | Neutral defaults, 7-day history, `load_recovery_imbalance` | Matches Firestore serving |
| **v2.1** | Drop `chronic_load_7d`; align ACWR to 7d-only baseline | Removes redundant load column; train/serve parity |
| **final** | **35** features, **XGBoostDeep**, threshold **0.18** | Re-train after `data_generator.py` + `run_pipeline.py` |

---
## Part 5 — Model selection (core section)

### The problem
"Best accuracy" is not enough for injury **prevention** — **Recall matters more than Precision**.  
Also: a **fixed threshold** (e.g. 0.40) is **misleading** — one model can look best only because of the threshold.

### Our selection algorithm (`train_model.py`)

```
1. Train 11 models (LogReg, RF, ExtraTrees, XGBoost variants…)
2. Threshold sweep: 0.08 → 0.60 (step 0.02)
3. Per model — best operating point:
   • Tier 0: Recall≥85%, FPR≤70%, Precision≥13%, F1≥22%
   • Tier 1: Recall≥80%, FPR≤70%
   • Tier 2: fallback — minimize FPR, maximize Recall
4. pick_best_model — rank: Tier → FPR → Recall → ROC-AUC → Brier
5. validate_metrics.py + model_loader.py — gates before promote / inference
```

### Winner: **XGBoostDeep @ 0.18**
- Recall **86.6%** · FPR **65.1%** · ROC-AUC **0.723** · Brier **0.115**
- Code: `ML_model/train_model.py` → `pick_best_model()`, `select_operating_threshold_for_model()`

### 5.1 — Why fixed threshold 0.40 misleads

- `model_comparison.csv` — metrics at **fixed threshold 0.40** (legacy)
- `threshold_sweep.csv` — **optimal threshold per model** under Recall-first policy

**Takeaway:** ExtraTrees looks like the "winner" @ 0.40, but @ 0.18 it flags ~100% of days (FPR≈100%) — not usable.

In [ ]:
# 5.1 — Fixed threshold 0.40 vs operating point
comparison_fixed = pd.read_csv(RUN_DIR / "model_comparison.csv")
best_ops = pd.read_csv(RUN_DIR / "best_operating_points.csv")
sweep = pd.read_csv(RUN_DIR / "threshold_sweep.csv")

KEY_MODELS = ["ExtraTrees", "ExtraTreesTuned", "XGBoostDeep", "XGBoostRaw", "LogisticRegression"]

fixed_view = (
    comparison_fixed[comparison_fixed["Model"].isin(KEY_MODELS)]
    .sort_values("Recall@Threshold", ascending=False)
    .rename(columns={"Recall@Threshold": "Recall@0.40", "FPR@Threshold": "FPR@0.40"})
)

def metrics_at_threshold(model: str, threshold: float) -> dict:
    row = sweep[(sweep["Model"] == model) & (np.isclose(sweep["Threshold"], threshold))]
    if row.empty:
        return {}
    r = row.iloc[0]
    return {"Model": model, "Threshold": threshold, "Recall": r["Recall"], "FPR": r["FPR"], "Precision": r["Precision"]}

op_rows = []
for model in KEY_MODELS:
    if model == WINNER:
        th = OPERATING_THRESHOLD
    else:
        sub = best_ops[best_ops["Model"] == model]
        th = float(sub.iloc[0]["Threshold"]) if not sub.empty else 0.40
    row = metrics_at_threshold(model, th)
    if row:
        op_rows.append(row)
operating_view = pd.DataFrame(op_rows)

display(Markdown("#### @ fixed threshold 0.40 — who looks like the winner? (misleading)"))
display(fixed_view[["Model", "Recall@0.40", "FPR@0.40", "ROC-AUC"]].style.format({"Recall@0.40": "{:.1%}", "FPR@0.40": "{:.1%}", "ROC-AUC": "{:.3f}"}))

display(Markdown("#### Operating points under policy"))
display(operating_view.sort_values("Recall", ascending=False).style.format({"Threshold": "{:.2f}", "Recall": "{:.1%}", "FPR": "{:.1%}", "Precision": "{:.1%}"}))

at_018 = sweep[(sweep["Threshold"] == OPERATING_THRESHOLD) & (sweep["Model"].isin(["ExtraTrees", WINNER]))]
display(Markdown(f"#### @ {OPERATING_THRESHOLD} — why not ExtraTrees?"))
display(at_018[["Model", "Recall", "FPR", "Precision"]].style.format({"Recall": "{:.1%}", "FPR": "{:.1%}", "Precision": "{:.1%}"}))

fig, ax = plt.subplots(figsize=(10, 5))
for model, color, ls in [(WINNER, "#2980b9", "-"), ("ExtraTrees", "#e74c3c", "--"), ("ExtraTreesTuned", "#e67e22", "-.")]:
    sub = sweep[sweep["Model"] == model]
    ax.plot(sub["FPR"], sub["Recall"], ls, color=color, label=model, lw=2)
ax.axhline(policy["recall_min"], color="green", ls=":", label=f"Recall ≥ {policy['recall_min']:.0%}")
ax.axvline(policy["fpr_max_operating"], color="red", ls=":", label=f"FPR ≤ {policy['fpr_max_operating']:.0%}")
ax.scatter([winner_metrics["FPR@Threshold"]], [winner_metrics["Recall@Threshold"]], s=120, c="#2980b9", zorder=5, edgecolors="black")
ax.set_xlabel("FPR"); ax.set_ylabel("Recall"); ax.set_title("Recall–FPR tradeoff — why XGBoostDeep @ operating point?")
ax.legend(fontsize=8); ax.set_xlim(0, 1.05); ax.set_ylim(0, 1.05)
plt.tight_layout(); plt.show()

### 5.2 — Winner metrics (XGBoostDeep @ operating threshold)

Numbers saved in `run_manifest.json` and enforced by production gates.

In [ ]:
metrics_display = pd.Series({
    "Recall @ operating threshold": winner_metrics["Recall@Threshold"],
    "Precision": winner_metrics["Precision@Threshold"],
    "F1": winner_metrics["F1@Threshold"],
    "FPR": winner_metrics["FPR@Threshold"],
    "ROC-AUC": winner_metrics["ROC-AUC"],
    "Brier score": winner_metrics["BrierScore"],
    "Log loss": winner_metrics["LogLoss"],
})

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#3498db"] * len(metrics_display)
bars = ax.barh(metrics_display.index.astype(str), metrics_display.values, color=colors)
ax.axvline(policy["recall_min"], color="green", ls="--", lw=2, label=f"Recall target {policy['recall_min']:.0%}")
ax.axvline(policy["fpr_max_operating"], color="red", ls="--", lw=2, label=f"Max FPR {policy['fpr_max_operating']:.0%}")
ax.set_xlim(0, 1)
ax.set_title(f"Winner: {WINNER} @ threshold {OPERATING_THRESHOLD}")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()
display(metrics_display.map(lambda x: f"{x:.3f}"))

### 5.3 — All 11 models @ operating point

Merges `model_comparison.csv` (ROC-AUC, Brier) with `best_operating_points.csv` (Recall/FPR @ optimal threshold).

In [ ]:
comparison = pd.read_csv(RUN_DIR / "model_comparison.csv")
best_ops_full = pd.read_csv(RUN_DIR / "best_operating_points.csv")
if "sweep" not in dir():
    sweep = pd.read_csv(RUN_DIR / "threshold_sweep.csv")

merged = comparison.merge(best_ops_full, on="Model", how="left", suffixes=("_at_0.40", "_operating"))

show_cols = ["Model", "ROC-AUC", "BrierScore", "Threshold", "Recall", "Precision", "FPR"]
display_df = merged[show_cols].sort_values("Recall", ascending=False, na_position="last").reset_index(drop=True)
display_df["Selected"] = display_df["Model"].eq(WINNER).map({True: "✓ winner", False: ""})

display(Markdown("#### All 11 models — sorted by Recall @ operating point"))
display(display_df.style.format({
    "ROC-AUC": "{:.3f}", "BrierScore": "{:.3f}", "Threshold": "{:.2f}",
    "Recall": "{:.1%}", "Precision": "{:.1%}", "FPR": "{:.1%}",
}))

# Catalog of 11 models (quick reference when explaining)
catalog_11 = pd.DataFrame([
    ("LogisticRegression", "Baseline linear + scaler"),
    ("RandomForest", "300 trees, balanced"),
    ("RandomForestTuned", "450 trees, custom class weights"),
    ("ExtraTrees", "320 trees — looks best @ 0.40 only"),
    ("ExtraTreesTuned", "520 trees — close but Recall < 85% @ policy"),
    ("GradientBoosting", "sklearn GBM"),
    ("XGBoostCalibrated", "XGB + sigmoid calibration"),
    ("XGBoostCalibratedTuned", "Tuned XGB + calibration"),
    ("XGBoostRaw", "v1 candidate — high FPR"),
    ("XGBoostDeep", "✓ WINNER — deeper trees, scale_pos_weight=3.5"),
    ("XGBoostDeepCalibrated", "Deep + calibration — Recall too low @ 0.18"),
], columns=["Model", "Notes"])
display(Markdown("#### Catalog of 11 candidates"))
display(catalog_11)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
subset = display_df.dropna(subset=["Recall"]).head(8)
x = np.arange(len(subset))
w = 0.35
axes[0].bar(x - w/2, subset["Recall"], w, label="Recall", color="#e74c3c")
axes[0].bar(x + w/2, subset["FPR"], w, label="FPR", color="#95a5a6")
axes[0].axhline(policy["recall_min"], color="green", ls="--", alpha=0.6)
axes[0].axhline(policy["fpr_max_operating"], color="red", ls="--", alpha=0.6)
axes[0].set_xticks(x)
axes[0].set_xticklabels(subset["Model"], rotation=45, ha="right")
axes[0].set_title("Recall vs FPR @ operating point")
axes[0].legend(fontsize=8)
axes[1].bar(subset["Model"], subset["ROC-AUC"], color="#3498db")
axes[1].set_title("ROC-AUC"); axes[1].tick_params(axis="x", rotation=45)
axes[2].bar(subset["Model"], subset["BrierScore"], color="#9b59b6")
axes[2].set_title("Brier (lower is better)"); axes[2].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

---
## Part 6 — Operating threshold, calibration, UI bands

**Score** = P(injury **today**). Three UI bands:

| Level | Score | Typical injury-day rate (holdout) |
|-------|-------|-----------------------------------|
| Low | < 0.11 | ~5% |
| Medium | 0.11 – 0.18 | ~11% |
| **High** | **≥ 0.18** | **~37%** |

**When explaining:** use the threshold sweep chart — show how 0.18 was chosen (Recall≥85%, FPR≤70%).

In [ ]:
if "sweep" not in dir():
    sweep = pd.read_csv(RUN_DIR / "threshold_sweep.csv")
deep = sweep[sweep["Model"] == WINNER].copy()

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(deep["Threshold"], deep["Recall"], "o-", color="#e74c3c", label="Recall", lw=2)
ax1.plot(deep["Threshold"], deep["FPR"], "s-", color="#95a5a6", label="FPR", lw=2)
ax1.axvline(OPERATING_THRESHOLD, color="green", lw=2, ls="--", label=f"Selected threshold {OPERATING_THRESHOLD}")
ax1.axvline(0.11, color="#f1c40f", lw=1, ls=":", label="Medium 0.11")
ax1.axvline(0.4, color="orange", lw=1, ls=":", label="Legacy comparison 0.40")
ax1.axhline(policy["recall_min"], color="green", alpha=0.35, label=f"Recall target {policy['recall_min']:.0%}")
ax1.axhline(policy["fpr_max_operating"], color="red", alpha=0.35, label=f"Max FPR {policy['fpr_max_operating']:.0%}")
ax1.set_xlabel("Probability threshold"); ax1.set_ylabel("Recall / FPR")
ax1.set_title(f"Threshold sweep — {WINNER}")
ax1.legend(fontsize=8, loc="center left", bbox_to_anchor=(1.02, 0.5))
ax2 = ax1.twinx()
ax2.plot(deep["Threshold"], deep["Precision"], "^-", color="#3498db", alpha=0.7, label="Precision")
ax2.set_ylabel("Precision")
plt.tight_layout()
plt.show()

In [ ]:
cal = pd.read_csv(RUN_DIR / "calibration_curve_data.csv")
deep_cal = cal[cal["model"] == WINNER]

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax.plot(deep_cal["mean_predicted_risk"], deep_cal["fraction_positive"], "o-", color="#16a085", lw=2)
ax.set_xlabel("Mean predicted P(injury on day D)")
ax.set_ylabel("Observed injury-day rate")
ax.set_title(f"Calibration — Brier {winner_metrics['BrierScore']:.3f}")
ax.legend()
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

risk_bins = pd.DataFrame(manifest["risk_bins"])
fig, ax = plt.subplots(figsize=(8, 4))
labels = ["0–20%", "20–50%", "50–100%"]
ax.bar(labels, risk_bins["injury_rate"] * 100, color=["#2ecc71", "#f1c40f", "#e74c3c"])
ax.set_ylabel("Observed injury-day rate (%)")
ax.set_title("Score bin vs actual injury rate (holdout)")
for i, row in risk_bins.iterrows():
    ax.text(i, row["injury_rate"] * 100 + 1, f"n={int(row['samples']):,}", ha="center")
plt.tight_layout()
plt.show()

display(pd.DataFrame({
    "UI level": ["Low", "Medium", "High"],
    "Score (today)": ["< 0.11", "0.11 – 0.18", "≥ 0.18"],
    "Typical injury-day rate": ["~5%", "~11%", "~37%"],
}))

---
## Part 7 — Model evolution (v1 Raw → final Deep)

| | v1 XGBoostRaw @ 0.30 | final XGBoostDeep @ 0.18 |
|--|----------------------|---------------------------|
| **Issue** | High recall, FPR ~79% | Recall 86.6%, FPR 65% |
| **Gain** | — | Better ROC-AUC, Brier, calibration |

In [ ]:
evolution = pd.DataFrame({
    "Metric": ["Recall", "ROC-AUC", "Brier", "Log loss", "FPR", "Threshold"],
    "v1 XGBoostRaw": [0.903, 0.654, 0.200, 0.587, 0.790, 0.30],
    "v2 XGBoostDeep": [
        winner_metrics["Recall@Threshold"],
        winner_metrics["ROC-AUC"],
        winner_metrics["BrierScore"],
        winner_metrics["LogLoss"],
        winner_metrics["FPR@Threshold"],
        OPERATING_THRESHOLD,
    ],
})
evolution["Δ v2−v1"] = evolution["v2 XGBoostDeep"] - evolution["v1 XGBoostRaw"]
display(evolution.round(3))

fig, ax = plt.subplots(figsize=(9, 4))
metrics_to_plot = ["ROC-AUC", "Brier", "Log loss", "FPR"]
x = np.arange(len(metrics_to_plot))
v1_vals = [0.654, 0.200, 0.587, 0.790]
v2_vals = [winner_metrics["ROC-AUC"], winner_metrics["BrierScore"], winner_metrics["LogLoss"], winner_metrics["FPR@Threshold"]]
ax.bar(x - 0.2, v1_vals, 0.4, label="v1 Raw", color="#bdc3c7")
ax.bar(x + 0.2, v2_vals, 0.4, label="v2 Deep", color="#2980b9")
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.set_title("Model version comparison (lower better: Brier, log loss, FPR)")
ax.legend()
plt.tight_layout()
plt.show()

---
## Part 8 — Summary

| Topic | Result |
|-------|--------|
| **Prediction** | Injury risk **on day D** (shown as **today** in the app) |
| **Features** | **35** kept; **3** removed (2× std21 + redundant `chronic_load_7d`) |
| **Top drivers** | 7 features ≈ **55%** of importance — see Part 3 |
| **Model selection** | **11 candidates** → Recall-first policy → **XGBoostDeep** |
| **Operating threshold** | **0.18** (not 0.40) |
| **Metrics** | Recall **~87%**, ROC-AUC **0.72**, Brier **0.11** |
| **Gates** | `validate_metrics.py` + `model_loader.py` — block weak models |

### One-line summary
> We built a full ML pipeline: hazard simulation, **35** engineered features, benchmark of 11 models, and a Recall-first selection algorithm with threshold sweep. **XGBoostDeep @ 0.18** won because it meets the operating policy — not just because ROC-AUC is highest at a fixed threshold.

**Re-run pipeline:** `python ML_model/data_generator.py` → `train_model.py` → `validate_metrics.py` → `run_pipeline.py`